# EtherCAT — satu frame melewati setiap terminal

Master (`EthercatMaster`) memindai rak I/O simulasi di `VirtualEthercatSegment`: coupler, 8 input digital, 8 output digital, dan 4 input analog pada sebuah konveyor. Untuk jalur sungguhan pakai `UseInterface("eth1")` di port khusus (Linux: CAP_NET_RAW; Windows: Npcap). Master tetap hanya-baca (SAFE-OP) sampai `AllowWrites()`.

## Persiapan
> Bekerja dari hasil clone? Jalankan `dotnet pack -c Release -o artifacts/packages` di root repo lalu tambahkan
> `#i "nuget: <repo>/artifacts/packages"` sebelum baris `#r`.

In [ ]:
#r "nuget: IoTCom.Net, 0.29.0-preview.1"
#r "nuget: IoTCom.Net.Native.Modbus, 0.29.0-preview.1"

## Frame dan datagram
Frame Ethernet ber-EtherType 0x88A4 membawa datagram; masing-masing diakhiri working counter yang dinaikkan oleh slave.

In [ ]:
using System.Buffers.Binary;
using IoTCom.Net;
using IoTCom.Net.Protocols.EtherCat;

var frame = EcatFrame.Encode([
    EcatDatagram.Broadcast(EcatCommand.Brd, EcatRegister.Type, new byte[1]),
    EcatDatagram.Configured(EcatCommand.Fprd, 0x1002, EcatRegister.AlStatus, new byte[2]),
]);
Console.WriteLine(Convert.ToHexString(frame));
foreach (var f in EcatFrame.DescribeFields(frame)) Console.WriteLine($"  {f.Name,-12} {f.Value}");

## Pindai jalur
Baca broadcast menghitung slave, tulis posisional memberi alamat stasiun, dan EEPROM SII memberi tahu siapa mereka.

In [ ]:
var rack = new EthercatRackSimulator();
var master = EthercatMaster.Create(o => o.UseLink(rack.Segment));
foreach (var slave in await master.ScanAsync()) Console.WriteLine(slave);

## SAFE-OP: input hidup, output aman
Master memetakan setiap terminal ke satu process image (FMMU) dan membacanya dengan satu datagram logis per siklus.

In [ ]:
await master.RequestStateAsync(EcatState.SafeOp);
for (var i = 0; i < 20; i++) { rack.Step(0.05); await master.ExchangeAsync(); }
Console.WriteLine($"DI8 {Convert.ToString(master.InputsOf(master.Slaves[1])[0], 2).PadLeft(8, '0')}  (bit 2 e-stop OK, bit 3 guard closed)");
var ai = master.InputsOf(master.Slaves[3]);
Console.WriteLine($"air {BinaryPrimitives.ReadInt16LittleEndian(ai[4..]) / 100.0:0.00} bar, gearbox {BinaryPrimitives.ReadInt16LittleEndian(ai[2..]) / 100.0:0.0} °C");
Console.WriteLine($"CoE 0x1008: {System.Text.Encoding.ASCII.GetString(await master.SdoReadAsync(master.Slaves[3], 0x1008, 0))}");
Console.WriteLine(master.Statistics);
try { await master.RequestStateAsync(EcatState.Op); } catch (ReadOnlyModeException e) { Console.WriteLine(e.Message); }

## OP, kabel putus
Master kedua dengan AllowWrites() menjalankan konveyor; lalu kabel diputus setelah terminal input dan working counter berkurang.

In [ ]:
await master.DisposeAsync();
var engineer = EthercatMaster.Create(o => o.UseLink(rack.Segment).AllowWrites());
await engineer.ScanAsync();
await engineer.RequestStateAsync(EcatState.Op);
engineer.SetOutputBit(engineer.Slaves[2], 0, true);   // conveyor motor
for (var i = 0; i < 300; i++) { rack.Step(0.01); await engineer.ExchangeAsync(); }
var analog = engineer.InputsOf(engineer.Slaves[3]);
Console.WriteLine($"belt {BinaryPrimitives.ReadInt16LittleEndian(analog[6..]) / 100.0:0.0} m/min, wkc {engineer.Statistics.LastWorkingCounter}/{engineer.Statistics.ExpectedWorkingCounter}");
rack.Segment.BreakAfter = 2;
await engineer.ExchangeAsync();
Console.WriteLine($"cable cut: wkc {engineer.Statistics.LastWorkingCounter}/{engineer.Statistics.ExpectedWorkingCounter}");
rack.Segment.BreakAfter = null;
await engineer.DisposeAsync();
Console.WriteLine($"after closing the master: DO8 is in {EcatAlStatus.Name(rack.DigitalOut.State)}");

## Lebih lanjut
`iotcom ethercat interfaces`, `iotcom ethercat scan --if <port>`, `iotcom ethercat watch --sim`, `iotcom ethercat sdo 3 1008:00 --sim`, `iotcom ethercat demo`, *Rak I/O lewat EtherCAT* di Gallery, dan sampel EthercatRack. Lihat `docs/id/protocols/ethercat.md`.

*IoTCom.Net — dibuat oleh Gravicode Studios dipimpin oleh Kang Fadhil.*